In [17]:
!pip -q install requests beautifulsoup4 lxml pandas nltk scikit-learn sentence-transformers
import re, time, random, requests, pandas as pd, numpy as np
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
SEED = 42; random.seed(SEED); np.random.seed(SEED)

In [18]:
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; student-nlp-lab)"}
# tier: edu (high), near (society/politics: ambiguous), off (unrelated)
LISTINGS = [
    ("https://www.hespress.com/tag/التعليم",        "edu",  "hespress"),
    ("https://www.hespress.com/tag/الدخول-المدرسي", "edu",  "hespress"),
    ("https://www.hespress.com/tag/الجامعة",        "edu",  "hespress"),
    ("https://www.hespress.com/politique",          "near", "hespress"),
    ("https://www.hespress.com/societe",            "near", "hespress"),
    ("https://www.hespress.com/sport",              "off",  "hespress"),
    ("https://www.hespress.com/economie",           "off",  "hespress"),
    ("https://www.hespress.com/art-et-culture",     "off",  "hespress"),
    ("https://al3omk.com/tag/التعليم",              "edu",  "al3omk"),
    ("https://al3omk.com/category/sport",           "off",  "al3omk"),
    ("https://al3omk.com/category/economie",        "off",  "al3omk"),
]
PAGES, MAX_LINKS = 5, 40
ARTICLE_RE = re.compile(r"(-\d{5,}\.html$)|(/\d{5,}\.html$)")

def get(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=20); r.raise_for_status()
        return r.text
    except Exception as e:
        print("ERR", url, str(e)[:80]); return None

def article_links(listing):
    links, base = [], listing.rstrip("/")
    for p in range(1, PAGES + 1):
        html = None
        for u in ([listing] if p == 1 else [f"{base}/page/{p}", f"{base}?page={p}"]):
            html = get(u)
            if html: break
        if not html: break
        soup, found = BeautifulSoup(html, "lxml"), 0
        for a in soup.find_all("a", href=True):
            h = urljoin(listing, a["href"]).split("#")[0]
            if urlparse(h).netloc == urlparse(listing).netloc and ARTICLE_RE.search(h):
                links.append(h); found += 1
        if not found: break
        time.sleep(random.uniform(0.8, 1.6))
    return list(dict.fromkeys(links))[:MAX_LINKS]

AR = re.compile(r"[\u0600-\u06FF]")
def parse_article(url):
    html = get(url)
    if not html: return None
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style", "nav", "footer", "aside", "form"]): t.decompose()
    title = soup.find("h1") or soup.find("title")
    title = title.get_text(" ", strip=True) if title else ""
    box = (soup.select_one("div.article-content") or soup.select_one("div.entry-content")
           or soup.select_one("article") or soup)
    paras = [p.get_text(" ", strip=True) for p in box.find_all("p")]
    paras = [p for p in paras if len(p) > 40 and len(AR.findall(p)) / max(len(p), 1) > 0.5]
    text = (title + ". " + " ".join(paras)).strip()
    return text if len(text.split()) >= 60 else None

rows = []
for lst, tier, site in LISTINGS:
    links = article_links(lst)
    print(f"{tier:5s} {site:9s} {lst} -> {len(links)} links")
    for u in links:
        t = parse_article(u)
        if t: rows.append(dict(Text=t, url=u, tier=tier, site=site, group=u))
        time.sleep(random.uniform(0.4, 0.9))
raw = pd.DataFrame(rows).drop_duplicates("Text").reset_index(drop=True)
raw.to_csv("raw_news.csv", index=False, encoding="utf-8-sig")
print(raw.shape); print(raw.groupby(["site", "tier"]).size())

edu   hespress  https://www.hespress.com/tag/التعليم -> 40 links
edu   hespress  https://www.hespress.com/tag/الدخول-المدرسي -> 40 links
edu   hespress  https://www.hespress.com/tag/الجامعة -> 40 links
near  hespress  https://www.hespress.com/politique -> 40 links
near  hespress  https://www.hespress.com/societe -> 40 links
off   hespress  https://www.hespress.com/sport -> 40 links
off   hespress  https://www.hespress.com/economie -> 40 links
off   hespress  https://www.hespress.com/art-et-culture -> 40 links
edu   al3omk    https://al3omk.com/tag/التعليم -> 40 links
off   al3omk    https://al3omk.com/category/sport -> 40 links
off   al3omk    https://al3omk.com/category/economie -> 40 links
(349, 5)
site      tier
al3omk    edu      40
          off      33
hespress  edu      96
          near     70
          off     110
dtype: int64


In [19]:
API = "https://ar.wikipedia.org/w/api.php"
WH = {"User-Agent": "EducationRelevanceLab/2.0 (student; contact: you@example.com)"}
WIKI = {
  "edu":  ["التعليم في المغرب","بيداغوجيا","علم النفس التربوي","منهج دراسي","التعليم العالي",
           "تكنولوجيا التعليم","محو الأمية","التعليم الابتدائي","التعليم الثانوي","التعلم"],
  "near": ["الشباب","سوق العمل","الصحة العامة","الأسرة","الإنترنت"],
  "off":  ["كرة القدم","اقتصاد المغرب","السينما المغربية","السياحة في المغرب",
           "الطاقة المتجددة","الطيران","المطبخ المغربي","الزراعة في المغرب"],
}
def wiki_chunks(title, tier, size=250, min_w=120, max_chunks=6):
    try:
        r = requests.get(API, params=dict(action="query", prop="extracts", explaintext=1,
              redirects=1, titles=title, format="json"), headers=WH, timeout=20)
        ext = next(iter(r.json()["query"]["pages"].values())).get("extract", "")
    except Exception as e:
        print("ERR", title, e); return []
    w = re.sub(r"\s+", " ", ext).split(); out = []
    for i in range(0, len(w), size):
        if len(w[i:i+size]) >= min_w and len(out) < max_chunks:
            out.append(dict(Text=" ".join(w[i:i+size]), url="wiki:"+title, tier=tier,
                            site="wikipedia", group="wiki:"+title))
    time.sleep(0.4); return out
wrows = [c for tier, ts in WIKI.items() for t in ts for c in wiki_chunks(t, tier)]
raw = pd.concat([raw, pd.DataFrame(wrows)], ignore_index=True)
raw.to_csv("raw_all.csv", index=False, encoding="utf-8-sig")
print(raw.groupby(["site", "tier"]).size())

site       tier
al3omk     edu      40
           off      33
hespress   edu      96
           near     70
           off     110
wikipedia  edu      39
           near     20
           off      42
dtype: int64


In [20]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

QUERIES = [
    "التعليم في المغرب وإصلاح المنظومة التربوية",
    "المدرسة والتلاميذ والأساتذة والمناهج الدراسية",
    "الجامعة والطلبة والتعليم العالي والبحث العلمي",
    "الدخول المدرسي والامتحانات والهدر المدرسي",
    "التربية والتعلم والبيداغوجيا وتكوين المدرسين",
]

q = emb.encode(QUERIES, normalize_embeddings=True).mean(0)
q /= np.linalg.norm(q)

short = lambda t: " ".join(str(t).split()[:256])
E = emb.encode([short(t) for t in raw["Text"]], batch_size=32, normalize_embeddings=True, show_progress_bar=True)

sim = E @ q
prior = raw["tier"].map({"edu": 1.0, "near": 0.4, "off": 0.0}).fillna(0.0).values
lo, hi = np.percentile(sim, [2, 98])
raw["Score"] = (10 * (0.8 * np.clip((sim - lo) / (hi - lo), 0, 1) + 0.2 * prior)).round(1)

print(raw.groupby("tier")["Score"].describe().round(2))

pd.set_option("display.max_colwidth", 200, "display.width", 250)
border = raw[(raw.Score.between(3, 5)) | (raw.Score.between(7, 9))]
sample_df = border.sample(min(40, len(border)), random_state=1)

for i, r in sample_df.iterrows():
    print(f"[{i}] {r.Score:4.1f} ({r.tier}) {r.Text[:200]}\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

      count  mean   std  min   25%   50%   75%   max
tier                                                
edu   175.0  7.76  1.69  3.4  6.55  8.10  9.10  10.0
near   90.0  4.11  1.51  1.1  3.10  3.75  4.78   8.3
off   185.0  2.62  1.53  0.0  1.60  2.60  3.40   7.9
[70]  8.1 (edu) لحجمري يعلق على تحول حفل تخرج الطلبة الجامعيين إلى "كرنفال عبثي". في تلك اللحظة العابرة التي ارتدى فيها الطلبة أزياء التَّخرّج، لم يكن الزمن على وشك أن يتوقف، لكنه بدا وكأنه يتعثر في لا معقوليته، يفقد

[391]  3.8 (near) الأحياء الدقيقة، مسؤولي الصحة البيئية، اختصاصيي صحة الأسنان، أطباء التخسيس ومتخصصي التغذية، مراقبي الصحة، الأطباء البيطريين، مهندسي الصحة العامة، محاميي الصحة العامة، علماء الاجتماع، عمال تنمية المجتم

[82]  7.9 (edu) صرف الدفعة الثانية من المنح لطلبة الجامعات. أفاد مصدر من المكتب الوطني للأعمال الجامعية الاجتماعية والثقافية بأنه سيتم استئناف عملية صرف المنح الجامعية لفائدة الطلبة والطالبات. وأشار المصدر ذاته إلى أ

[114]  3.5 (near) شراكة أمنية مغربية إماراتية جديدة تكافح الجرائم والتهديدات ال

In [22]:
pd.set_option("display.max_colwidth", 200, "display.width", 250)
border = raw[(raw.Score.between(3, 5)) | (raw.Score.between(7, 9))].sample(min(40, len(raw)), random_state=1)
for i, r in border.iterrows():
    print(f"[{i}] {r.Score:4.1f} ({r.tier}) {r.Text[:200]}\n")

[70]  8.1 (edu) لحجمري يعلق على تحول حفل تخرج الطلبة الجامعيين إلى "كرنفال عبثي". في تلك اللحظة العابرة التي ارتدى فيها الطلبة أزياء التَّخرّج، لم يكن الزمن على وشك أن يتوقف، لكنه بدا وكأنه يتعثر في لا معقوليته، يفقد

[391]  3.8 (near) الأحياء الدقيقة، مسؤولي الصحة البيئية، اختصاصيي صحة الأسنان، أطباء التخسيس ومتخصصي التغذية، مراقبي الصحة، الأطباء البيطريين، مهندسي الصحة العامة، محاميي الصحة العامة، علماء الاجتماع، عمال تنمية المجتم

[82]  7.9 (edu) صرف الدفعة الثانية من المنح لطلبة الجامعات. أفاد مصدر من المكتب الوطني للأعمال الجامعية الاجتماعية والثقافية بأنه سيتم استئناف عملية صرف المنح الجامعية لفائدة الطلبة والطالبات. وأشار المصدر ذاته إلى أ

[114]  3.5 (near) شراكة أمنية مغربية إماراتية جديدة تكافح الجرائم والتهديدات العابرة للحدود. وقّع المدير العام للأمن الوطني ولمراقبة التراب الوطني، عبد اللطيف حموشي، والقائد العام لشرطة دبي، الفريق عبد الله خليفة المري

[58]  9.0 (edu) هجوم إلكتروني يربك الدخول المدرسي في فرنسا. عاد المدرسون في فرنسا إلى المؤسسات التعليمية، الاثنين، في دخول م

In [23]:
before = len(raw)
raw["Text"] = (raw["Text"].str.replace(r"={2,}", " ", regex=True)
                          .str.replace(r"\s+", " ", regex=True).str.strip())
raw = raw.drop_duplicates("Text").reset_index(drop=True)
print(f"removed {before - len(raw)} duplicate rows -> {len(raw)} left")
print(raw.groupby("tier")["Score"].describe().round(2))

removed 0 duplicate rows -> 450 left
      count  mean   std  min   25%   50%   75%   max
tier                                                
edu   175.0  7.76  1.69  3.4  6.55  8.10  9.10  10.0
near   90.0  4.11  1.51  1.1  3.10  3.75  4.78   8.3
off   185.0  2.62  1.53  0.0  1.60  2.60  3.40   7.9


In [24]:
# Wikipedia "near" pages (family, youth, internet, public health) are not education
m = (raw.site == "wikipedia") & (raw.tier == "near")
raw.loc[m, "Score"] = raw.loc[m, "Score"].clip(upper=3.5)

MANUAL = {
    # not education / incidental -> class 0
    "فرنسا تغلي على وقع احتجاجات": 2.0,
    "إطلاق نار بجامعة أمريكية": 3.0,
    "عاهل البحرين يثمن الأخوة": 2.0,
    "الملك يترأس جلسة عمل بشأن": 2.0,
    "أحزاب تطالب بإلغاء مجالس العمالات": 2.0,
    "الحكومة تضخ 600 مليون درهم": 2.0,
    "قيادة مركز حقوقي": 2.0,
    "بلفقيه يشيد بسرعة إعادة الكهرباء": 2.0,
    "المغرب يقوي طموح الدخول إلى نادي": 2.0,
    "القضاء يرفض شكاية البروفيسور": 2.0,
    "الكرونولوجية. ففي البداية": 2.0,
    "على الموروث الثقافي": 2.5,
    # education touched, not the main subject -> class 1
    "يثير مخاوف داخل المدارس": 5.5,
    "لحجمري يعلق على تحول حفل تخرج": 7.0,
    "المغاربة يتصدرون قائمة الطلاب الوافدين": 7.0,
    # core education -> class 2
    "وزير التربية يطمئن أطر التعليم الأولي": 8.5,
    "تعدّل شروط ترقية الأساتذة الباحثين": 8.0,
    "القانون الجديد يهدد الجامعة": 9.0,
    "مساحة التعلم تشير": 8.0,
    "التَّعَلُّم من الناحية اللغوية": 8.5,
}
hit = 0
for snip, s in MANUAL.items():
    mm = raw["Text"].str.contains(snip, regex=False)
    if mm.sum() == 0: print("NOT FOUND:", snip)
    raw.loc[mm, "Score"] = s; hit += int(mm.sum())
print("overrides applied to", hit, "rows from", len(MANUAL), "snippets")

overrides applied to 20 rows from 20 snippets


In [25]:
from sklearn.model_selection import GroupShuffleSplit
def discretize(s): return 0 if s < 4 else (1 if s < 8 else 2)
raw["label"] = raw["Score"].apply(discretize)
print(raw["label"].value_counts().sort_index()); print(pd.crosstab(raw["tier"], raw["label"]))

tr_idx, te_idx = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED).split(raw, raw["label"], groups=raw["group"]))
trv, test = raw.iloc[tr_idx], raw.iloc[te_idx]
a, b = next(GroupShuffleSplit(1, test_size=0.15, random_state=SEED).split(trv, trv["label"], groups=trv["group"]))
raw["split"] = "train"; raw.loc[trv.iloc[b].index, "split"] = "val"; raw.loc[test.index, "split"] = "test"
print(pd.crosstab(raw["split"], raw["label"]))

label
0    220
1    136
2     94
Name: count, dtype: int64
label    0   1   2
tier              
edu      4  81  90
near    58  28   4
off    158  27   0
label    0   1   2
split             
test    39  29  18
train  150  88  63
val     31  19  13


In [26]:
import nltk
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
nltk.download("stopwords", quiet=True)

DIAC = re.compile(r"[\u0617-\u061A\u064B-\u0652]"); TAT = re.compile(r"\u0640")
def normalize_ar(t):
    t = DIAC.sub("", str(t)); t = TAT.sub("", t)
    t = re.sub("[إأآ]", "ا", t); t = t.replace("ى", "ي").replace("ة", "ه")
    t = re.sub(r"(.)\1{2,}", r"\1", t)
    t = re.sub(r"[^\u0621-\u064A\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

STOP = {normalize_ar(w) for w in stopwords.words("arabic")}
STOP |= {normalize_ar(w) for w in ["هسبريس","اليوم","قال","وقال","كما","حيث","اضافه","بعد","عبر",
        "خلال","ذلك","هذا","هذه","التي","الذي","ايضا","وفق","حسب","العلم","الرباط"]}
stem = ISRIStemmer().stem
try:
    !pip -q install qalsadi
    import qalsadi.lemmatizer as _q
    lemma = _q.Lemmatizer().lemmatize; print("qalsadi OK")
except Exception as e:
    lemma = None; print("no lemmatizer, stems only:", str(e)[:60])

def preprocess(t):
    toks = [w for w in normalize_ar(t).split() if w not in STOP and len(w) > 1]
    return toks, [stem(w) for w in toks], ([lemma(w) for w in toks] if lemma else [])

res = raw["Text"].apply(preprocess)
raw["tokens"] = res.apply(lambda r: " ".join(r[0]))
raw["stems"]  = res.apply(lambda r: " ".join(r[1]))
raw["lemmas"] = res.apply(lambda r: " ".join(r[2]))
raw["n_tokens"] = raw["tokens"].str.split().str.len()
print(raw["n_tokens"].describe().round(1))

qalsadi OK
count     450.0
mean      341.5
std       268.7
min        59.0
25%       181.2
50%       255.0
75%       434.0
max      2251.0
Name: n_tokens, dtype: float64


In [28]:
raw[["Text", "Score"]].to_csv("dataset_text_score.csv", index=False, encoding="utf-8-sig")  # lab format
raw.to_csv("dataset_education.csv", index=False, encoding="utf-8-sig")                        # for Notebook 2
print(raw.shape)

(450, 12)
